# Price Elasticity Walkthrough

How the workbench recovers a per-event own-price elasticity from a synthetic panel, and turns that into a contribution-margin-maximizing recommended price.

Run `python -m backend.seed` before running this notebook.

In [ ]:
import sys
sys.path.append('..')

from backend.database import SessionLocal
from backend.models import Event
from backend.services.elasticity import fit_log_log, revenue_maximizing_price
from backend.services.metrics import avg_ticket_price, contribution_margin, gross_revenue

db = SessionLocal()
events = db.query(Event).all()
print(f'Loaded {len(events)} events.')

## Fit elasticities for every event

In [ ]:
import pandas as pd
rows = []
for e in events:
    prices = [h.avg_price for h in e.historical_pricing]
    qtys = [h.tickets_sold for h in e.historical_pricing]
    fit = fit_log_log(prices, qtys)
    cp = avg_ticket_price(e.tickets)
    rec, grid = revenue_maximizing_price(fit, cp)
    rows.append({
        'city': e.venue.city,
        'beta_1': round(fit.beta_1, 3),
        'r2': round(fit.r_squared, 3),
        'n': fit.sample_size,
        'current_price': round(cp, 2),
        'recommended_price': round(rec, 2),
        'revenue': round(gross_revenue(e.tickets), 2),
        'contribution': round(contribution_margin(e.tickets), 2),
    })
df = pd.DataFrame(rows).sort_values('beta_1')
df

## Plot one event's (price, quantity) panel with the fitted curve


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

event = next(e for e in events if e.venue.city == 'Chicago')
prices = np.array([h.avg_price for h in event.historical_pricing])
qtys = np.array([h.tickets_sold for h in event.historical_pricing])
fit = fit_log_log(prices, qtys)

grid = np.linspace(prices.min(), prices.max(), 60)
predicted = np.exp(fit.beta_0) * grid ** fit.beta_1

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(prices, qtys, alpha=0.6, label='Observed')
ax.plot(grid, predicted, color='tomato', linewidth=2,
        label=f'Fit: log q = {fit.beta_0:.2f} + {fit.beta_1:.2f}·log p, R²={fit.r_squared:.2f}')
ax.set_xlabel('Price ($)')
ax.set_ylabel('Tickets sold / day')
ax.set_title(f"{event.event_name} — log-log elasticity fit")
ax.legend()
plt.tight_layout()

## Grid-scan the contribution-margin surface


In [ ]:
cp = avg_ticket_price(event.tickets)
rec, grid = revenue_maximizing_price(fit, cp, variable_cost_per_ticket=25.0)
grid_df = pd.DataFrame(grid)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(grid_df['price'], grid_df['predicted_revenue'], label='Revenue', linestyle='--')
ax.plot(grid_df['price'], grid_df['predicted_contribution_margin'], label='Contribution margin')
ax.axvline(cp, color='#9aa5bd', linestyle=':', label='Current price')
ax.axvline(rec, color='#3fbfad', label='Recommended price')
ax.set_xlabel('Price ($)')
ax.set_ylabel('Value ($)')
ax.set_title(f"{event.event_name} — bounded grid search")
ax.legend()
plt.tight_layout()